In [32]:
from pyspark.sql import SparkSession
import time

In [33]:
spark = SparkSession.builder \
    .appName("Apex Financial Data Ingestion") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

In [34]:
path = "c:/Users/u/Desktop/Repositories/Apex_financial/data/gold/"
fact_transactions = spark.read.parquet(f"{path}fact_transactions.parquet")

fact_transactions.show(5)

+--------------+-----------+-----------+-----------+---------+-------------------+------+------------+----------+------------+---------------+---------------+------------+--------+
|transaction_id|customer_id|    card_id|merchant_id|device_id|          timestamp|amount|product_type| card_type|payment_type|customer_region|merchant_region|has_identity|is_fraud|
+--------------+-----------+-----------+-----------+---------+-------------------+------+------------+----------+------------+---------------+---------------+------------+--------+
|     T00000001|    C001131|CARD0001498|    M000467| D0001467|2026-05-26 21:43:58| 21.09|           W|      visa|       debit|           NULL|         Kisumu|       false|       0|
|     T00000002|    C001769|CARD0002353|    M000237| D0002277|2026-05-29 08:35:28| 36.01|           W|      visa|      credit|           NULL|         Kisumu|       false|       0|
|     T00000003|    C001770|CARD0002356|    M000752| D0002278|2026-04-07 14:35:18|158.71|      

In [35]:
transactions_100k = (
    fact_transactions
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
)

In [36]:
print("the count of transactions_100k is: ", transactions_100k.count())
print("the count of fact_transactions is: ", fact_transactions.count())


from pyspark.sql import functions as F
import time

rounds = [1,2,3]
partitions = [4, 6, 8, 10, 12, 14, 16]

for i in rounds:
    print(f"Round {i}:")
    for p in partitions:
        spark.conf.set("spark.sql.shuffle.partitions", str(p))

        start = time.time()

        result = (
            transactions_100k
            .groupBy("customer_id")
            .agg(
                F.count("transaction_id").alias("transaction_count"),
                F.sum("amount").alias("total_amount"),
                F.avg("amount").alias("avg_transaction_amount")
            )
        )

        result.count()  # action → actually executes the Spark job

        elapsed = time.time() - start

        print(f"{p} partitions: {elapsed:.3f} seconds")

the count of transactions_100k is:  100000
the count of fact_transactions is:  10000
Round 1:
4 partitions: 0.394 seconds
6 partitions: 0.346 seconds
8 partitions: 0.585 seconds
10 partitions: 0.842 seconds
12 partitions: 1.052 seconds
14 partitions: 0.821 seconds
16 partitions: 0.768 seconds
Round 2:
4 partitions: 0.711 seconds
6 partitions: 0.623 seconds
8 partitions: 0.996 seconds
10 partitions: 0.629 seconds
12 partitions: 0.630 seconds
14 partitions: 0.661 seconds
16 partitions: 0.831 seconds
Round 3:
4 partitions: 0.638 seconds
6 partitions: 0.780 seconds
8 partitions: 0.561 seconds
10 partitions: 0.708 seconds
12 partitions: 0.693 seconds
14 partitions: 0.710 seconds
16 partitions: 0.750 seconds
